<a href="https://colab.research.google.com/github/MayerT1/AGU_Session_Hunter/blob/main/AGU_Session_Hunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# AGU26 Session Scraper + Keyword Filter (Colab-ready)
# ============================================================
!pip install -q requests beautifulsoup4 pandas

import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ---- 1. Category pages ----
CATEGORIES = {
    "Atmospheric and Space Electricity": "AE.html",
    "Atmospheric Sciences": "A.html",
    "Biogeosciences": "B.html",
    "Cryosphere": "C.html",
    "Earth and Planetary Surface Processes": "EP.html",
    "Education": "ED.html",
    "Geodesy": "G.html",
    "GeoHealth": "GH.html",
    "Geomagnetism, Paleomagnetism and Electromagnetism": "GP.html",
    "Global Environmental Change": "GC.html",
    "Hydrology": "H.html",
    "Informatics": "IN.html",
    "Mineral and Rock Physics": "MR.html",
    "Natural Hazards": "NH.html",
    "Near Surface Geophysics": "NS.html",
    "Nonlinear Geophysics": "NG.html",
    "Ocean Sciences": "OS.html",
    "Paleoceanography and Paleoclimatology": "PP.html",
    "Planetary Sciences": "P.html",
    "Science and Society": "SY.html",
    "Seismology": "S.html",
    "SPA-Aeronomy": "SA.html",
    "SPA-Magnetospheric Physics": "SM.html",
    "SPA-Solar and Heliospheric Physics": "SH.html",
    "Study of Earth's Deep Interior": "DI.html",
    "Tectonophysics": "T.html",
    "Union Sessions": "U.html",
    "Volcanology, Geochemistry and Petrology": "V.html",
}

BASE = "https://studio.m-anage.com/agu/agu26/webprogrampreliminary/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AGU-session-scraper/1.0)"}

# Regex matching each session link: <a href="SessionNNNNNN.html">Title</a>
SESSION_LINK_RE = re.compile(
    r'<a[^>]+href="(Session(\d+)\.html)"[^>]*>(.*?)</a>', re.IGNORECASE | re.DOTALL
)

def strip_tags(html_fragment):
    return BeautifulSoup(html_fragment, "html.parser").get_text(" ", strip=True)

def parse_category(cat_name, page_file):
    url = urljoin(BASE, page_file)
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    html = resp.text

    matches = list(SESSION_LINK_RE.finditer(html))
    rows = []
    for i, m in enumerate(matches):
        href, sess_num, raw_title = m.group(1), m.group(2), m.group(3)
        title = strip_tags(raw_title)

        # detail block = raw html between this link and the next session link
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(html)
        detail_html = html[start:end]
        detail_text = strip_tags(detail_html)

        # trim footer boilerplate off the last session's detail block
        cutoff = detail_text.find("Browse Sessions")
        if cutoff != -1:
            detail_text = detail_text[:cutoff]

        rows.append({
            "category": cat_name,
            "session_number": sess_num,
            "title": title,
            "url": urljoin(BASE, href),
            "detail_text": detail_text,
        })
    return rows

# ---- 2. Scrape all categories ----
all_rows = []
for cat_name, page_file in CATEGORIES.items():
    try:
        rows = parse_category(cat_name, page_file)
        all_rows.extend(rows)
        print(f"{cat_name}: {len(rows)} sessions")
    except Exception as e:
        print(f"FAILED {cat_name} ({page_file}): {e}")
    time.sleep(0.5)  # be polite to the server

df = pd.DataFrame(all_rows)
print(f"\nTotal sessions scraped: {len(df)}")

# ---- 3. Keyword filter ----
KEYWORDS = [
    "Foundational",
    "AI",
    "Geo-AI",
    "Generative AI",
    "NASA",
    "Huntsville",
    "Large Earth Model",
]

def make_keyword_pattern(keywords):
    parts = [r'\b' + re.escape(kw) + r'\b' for kw in keywords]
    return re.compile("|".join(parts), re.IGNORECASE)

pattern = make_keyword_pattern(KEYWORDS)

def find_matches(row):
    text = f"{row['title']} {row['detail_text']}"
    hits = sorted(set(m.group(0) for m in pattern.finditer(text)))
    return hits

df["matched_keywords"] = df.apply(find_matches, axis=1)
df["keyword_hit"] = df["matched_keywords"].apply(lambda x: len(x) > 0)

filtered_df = df[df["keyword_hit"]].copy()
print(f"Sessions matching keywords: {len(filtered_df)}")

# ---- 4. Save outputs ----
df.drop(columns=["detail_text"]).to_csv("agu26_all_sessions.csv", index=False)
filtered_df.drop(columns=["detail_text"]).to_csv("agu26_keyword_matches.csv", index=False)

print("\nSaved: agu26_all_sessions.csv, agu26_keyword_matches.csv")
filtered_df[["category", "session_number", "title", "url", "matched_keywords"]]

Atmospheric and Space Electricity: 8 sessions
Atmospheric Sciences: 122 sessions
Biogeosciences: 106 sessions
Cryosphere: 34 sessions
Earth and Planetary Surface Processes: 42 sessions
Education: 30 sessions
Geodesy: 25 sessions
GeoHealth: 28 sessions
Geomagnetism, Paleomagnetism and Electromagnetism: 7 sessions
Global Environmental Change: 127 sessions
Hydrology: 163 sessions
Informatics: 39 sessions
Mineral and Rock Physics: 16 sessions
Natural Hazards: 53 sessions
Near Surface Geophysics: 16 sessions
Nonlinear Geophysics: 9 sessions
Ocean Sciences: 34 sessions
Paleoceanography and Paleoclimatology: 24 sessions
Planetary Sciences: 30 sessions
Science and Society: 57 sessions
Seismology: 30 sessions
SPA-Aeronomy: 25 sessions
SPA-Magnetospheric Physics: 26 sessions
SPA-Solar and Heliospheric Physics: 43 sessions
Study of Earth's Deep Interior: 14 sessions
Tectonophysics: 19 sessions
Union Sessions: 37 sessions
Volcanology, Geochemistry and Petrology: 25 sessions

Total sessions scraped

,category,session_number,title,url,matched_keywords
0,Atmospheric and Space Electricity,280576,"Advancements in Lightning Meteorology, Climato...",https://studio.m-anage.com/agu/agu26/webprogra...,"[Huntsville, NASA]"
5,Atmospheric and Space Electricity,280118,Energetic Radiation from Lightning and Thunder...,https://studio.m-anage.com/agu/agu26/webprogra...,[Huntsville]
8,Atmospheric Sciences,282333,"AERONET and Sun Photometry: Retrievals, Calibr...",https://studio.m-anage.com/agu/agu26/webprogra...,[NASA]
9,Atmospheric Sciences,279311,AI and High Resolution Weather Forecasting,https://studio.m-anage.com/agu/agu26/webprogra...,[AI]
10,Atmospheric Sciences,279530,AI and Machine Learning for Improved Subseason...,https://studio.m-anage.com/agu/agu26/webprogra...,[AI]
...,...,...,...,...,...
1149,Union Sessions,281438,From AI to climate modeling to economic impact...,https://studio.m-anage.com/agu/agu26/webprogra...,[AI]
1151,Union Sessions,283090,NISAR One Year After the Start of its Science ...,https://studio.m-anage.com/agu/agu26/webprogra...,[NASA]
1154,Union Sessions,280799,"Prediction, Prevention, and Participation: Lin...",https://studio.m-anage.com/agu/agu26/webprogra...,[NASA]
1159,Union Sessions,281273,Synergistic developments at the Geoscience-AI ...,https://studio.m-anage.com/agu/agu26/webprogra...,[AI]
